In [2]:
!pip install gspread oauth2client

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name('C:\\Users\\Pratham Thakkar\\Documents\\celtechtask-7a6c5c8a9f8b.json', scope)
client = gspread.authorize(creds)
mentorsheet = client.open("Mentor and Startup Sample Database").worksheet("Mentor")
startupsheet = client.open("Mentor and Startup Sample Database").worksheet("Startup")
mentordata = mentorsheet.get_all_records()
startupdata = startupsheet.get_all_records()
mentordf = pd.DataFrame(mentordata)
startupdf = pd.DataFrame(startupdata)
def splitfields(df, columns):
    for column in columns:
        df[column] = df[column].apply(lambda x: x.split('?') if isinstance(x, str) else [])
    return df
mentordf = splitfields(mentordf, ['Mentor Code', 'Verticals', 'Horizontals', 'Business Model', 'Functions of Expertise'])
startupdf = splitfields(startupdf, ['Startup code', 'Vertical', 'Horizontal', 'Business Model', 'Business functions', 'Expert Preference 1', 'Expert Preference 2', 'Expert Preference 3', 'Expert Preference 4', 'Expert Preference 5'])


In [4]:
def calculatescore(startup, mentor):
    score = 0
    if any(i in mentor['Verticals'] for i in startup['Vertical']):
        score += 10  
    if any(j in mentor['Horizontals'] for j in startup['Horizontal']):
        score += 10  
    if any(k in mentor['Business Model'] for k in startup['Business Model']):
        score += 5 
    if any(w in mentor['Functions of Expertise'] for w in startup['Business functions']):
        score += 3  
    preferences = [startup['Expert Preference 1'], startup['Expert Preference 2'], startup['Expert Preference 3'], startup['Expert Preference 4'], startup['Expert Preference 5']]
    
    if mentor['Mentor Code'] == ['Expert Preference 1']:
        score += 10
    elif mentor['Mentor Code'] == ['Expert Preference 2']:
        score += 9
    elif mentor['Mentor Code'] == ['Expert Preference 3']:
        score += 8
    elif mentor['Mentor Code'] == ['Expert Preference 4']:
        score += 7
    elif mentor['Mentor Code'] == ['Expert Preference 5']:
        score += 6

    return score


In [6]:
compatibilityscores = {}

for _, startup in startupdf.iterrows():
    scores = []
    for _, mentor in mentordf.iterrows():
        score = calculatescore(startup, mentor)
        scores.append((mentor['Mentor Code'], score))
    topmentors = sorted(scores, key=lambda x: x[1], reverse=True)[:2]
    startupcode = startup['Startup code'] 
    if isinstance(startupcode, list):
        startupcode = startupcode[0] 
    if startupcode:  
        compatibilityscores[startupcode] = topmentors

In [7]:
results = []

for startupcode, mentors in compatibilityscores.items():
    for mentorcode, score in mentors:
            results.append({'Startup Code': startupcode, 'Mentor Code': mentorcode, 'Score': score})

resultdf = pd.DataFrame(results)
resultdf.to_csv('mentor_results.csv', index=False)